<a href="https://colab.research.google.com/github/JCARNEIROX/IA367-Aprendizado-Reforco/blob/main/Lista2_Ex6_RA239738_RA256389.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**IA368FF - Aprendizado por Reforço**  
1º Semestre de 2024  
Prof. Denis Fantinato

In [ ]:
import numpy as np

# Representando nossos problemas

Para representar um problema precisamos definir:

- Estado `st`
- Ações `actions`
- Função de transição `moves`
- Função de custo
- Estados inicial e final `s0, goalSt`

# Problema 8 Puzzle

Para esse problema a representação de estado será uma tupla `(sol, moves)` composta de duas listas, a primeira representando o estado final com a solução e a segunda o conjunto de ações para chegar nesse estado.

O estado inicial é uma permutação aleatória do quebra-cabeça e um conjunto vazio de jogadas: `s0 = ([[8,7,6],[5,0,1],[2,3,4]],[])` e cada ação altera a permutação do quebra-cabeça e acrescenta uma jogada.

O bloco abaixo define as transições de estado do problema 8 Puzzle. Implementa uma ação, modificando o estado. **Não modificar**.

In [ ]:
# Utility functions
empty = lambda x: len(x) == 0

def addTuple(x, y):
    return (x[0] + y[0], x[1] + y[1])

def absDiff(x, y):
    return (abs(x[0]-y[0]), abs(x[1]-y[1]))

def dist(v, x, y):
    x, y = absDiff( findNum(v,x), findNum(v,y) )
    return x+y

def findNum(v, x):
    for i, xi in enumerate(x):
        for j, xij in enumerate(xi):
            if(xij == v): return (i,j)

findZero = lambda x: findNum(0, x)

argmin = lambda ls: sorted(ls, key=lambda x: x[0])[0][1]

def inBound(x):
    if 0 <= x <= 2:
        return True
    return False

# Swap the values of a given state
def swap(val, s):
    new_s = [si.copy() for si in s]  # taking proper care about pass by reference
    x1, y1 = findNum(val, s)
    x2, y2 = findZero(s)
    new_s[x1][y1] = 0
    new_s[x2][y2] = val
    return new_s

# Maps a Move to a coordinate
move2coord = {"Up": (-1,0), "Down": (1,0), "Left": (0, -1), "Right": (0,1)}
pos = lambda mv: move2coord[mv]

# Update the state
def move(direc, st):
    s, t = st
    newX, newY = addTuple(findZero(s), pos(direc))
    if inBound(newX) and inBound(newY):
        val = s[newX][newY]
    else:
        val = 0
    new_s   = swap(val, s)
    return (new_s, t + [direc])

# Goal state
goalSt = [[1,2,3],[4,5,6],[7,8,0]]

# Check if it is a goal state
def isGoal( st ):
    s, t = st
    return s == goalSt

# For this puzzle, every state is feasible
def feasible(x):
    return True

# perform a move
def moves(sts):
    new_sts = []
    for st in sts:
        choices = move2coord.keys()
        for c in choices:
            new_sts.append(move(c, st))
    return new_sts




O bloco abaixo define a Busca em Largura ou, em inglês, Breadth-first search. **Não modificar**.


In [ ]:
# Breadth-first search
def bfs( sts ):
    if len(sts) == 0:
        print("Couldn't find a feasible solution")
        return ([],[]), 0
    goal    = list(filter(isGoal, sts))
    n       = 0
    while len(goal) == 0:
        sts  = list(filter(feasible, moves(sts)))
        goal = list(filter(isGoal, sts))
        n    = n+len(sts)
    return goal[0], n


O trecho seguinte define as heurísticas e o algoritmo A\*. **Não modificar**.

As heurísticas utilizadas pelo A* são:

*   f1 = soma das distâncias horizontais e verticais de cada peça até sua posição alvo.
*   f2 = quantidade de peças fora do lugar.
*   f3 = max(f1,f2)


In [ ]:
# A* search
def f1(st):
    s, t = st
    return len(t) + sum([dist(v, s, goalSt) for v in range(1,9)])

def f2(st):
    s, t = st
    return len(t) + sum([s[i][j] != goalSt[i][j]
                        for i in range(3)
                        for j in range(3) ]) - 1

def f3(st):
    return max(f1(st), f2(st))

def astar( sts, f ):
    if len(sts) == 0:
        print("Couldn't find a feasible solution")
        return ([],[]), 0

    goal    = list(filter(isGoal, sts))
    n       = 0
    sold = sts[0]
    while len(goal) == 0:
        s       = argmin(zip(map(f, sts), sts))
        sts = [si for si in sts if si[0] != s[0]]
        sts += moves([s])
        sold = s
        goal    = list(filter(isGoal, sts))
        n       = n + len(moves([s]))
    return goal[0], n


**MODIFICAR:** Escolha o estado inicial abaixo. Usar o formato:
`
s0 = [[1,2,3],[4,5,6],[7,8,0]]
`, em que a ordem dos tiles deve ser alterada.




In [ ]:
s0 = [[, , ], [, , ], [, , ]]

Aplica cada um dos algoritmos (Busca em Largura e A\* para cada uma das heurísticas). **Não modificar**.

In [ ]:
sB, nB = bfs([(s0,[])])
print(f'Busca em largura atingiu o estado {sB[0]} com a sequência {sB[1]} em {nB} jogadas\n\n')

sA1, nA1 = astar([(s0,[])], f1)
print(f'A*-f1 atingiu o estado {sA1[0]} com a sequência {sA1[1]} em {nA1} jogadas\n\n')

sA2, nA2 = astar([(s0,[])], f2)
print(f'A*-f2 atingiu o estado {sA2[0]} com a sequência {sA2[1]} em {nA2} jogadas\n\n')

sA3, nA3 = astar([(s0,[])], f3)
print(f'A*-f3 atingiu o estado {sA3[0]} com a sequência {sA3[1]} em {nA3} jogadas\n\n')